# Multi-class Mask Generator

Opens a **napari** viewer with the **multi-class annotation** widget for drawing region polygons.

Three region classes, each with its own colour-coded Shapes layer:

| ID | Class | Colour |
|----|-------|--------|
| 1  | nucleus | cyan |
| 2  | full_cell | green |
| 3  | lysosome | red |

- Select the active class via radio buttons (switches the active Shapes layer)
- Draw polygons, switch class, and repeat
- Navigate images with **Prev / Next** (auto-saves on switch)

**On Save**, everything for an image lands in the same folder under
`OUTPUT_PATH`, mirroring its subfolder under `INPUT_PATH`:

```
<INPUT_PATH>/<rel>/<stem>.tif         ->   raw image
<OUTPUT_PATH>/<rel>/<stem>_annotations.json   (COCO-style polygons)
<OUTPUT_PATH>/<rel>/<stem>_full_cell.tif      (per-class binary masks,
<OUTPUT_PATH>/<rel>/<stem>_nucleus.tif         uint8, 0/255, native shape)
<OUTPUT_PATH>/<rel>/<stem>_lysosome.tif
<OUTPUT_PATH>/<rel>/<stem>_cytosol.tif        (derived = full_cell AND NOT nucleus)
<OUTPUT_PATH>/<rel>/<stem>_manifest.json      (per-class n_instances, n_pixels)
```

## Configuration

- **`INPUT_PATH`**: path to a single TIFF image **or** a folder of TIFFs.
- **`OUTPUT_PATH`**: folder where all outputs are written (annotation JSON,
  per-class TIF masks, manifest). Each image's outputs land in the same
  subfolder relative to `INPUT_PATH` (so e.g. an image at
  `INPUT_PATH/Oxygen/600V_O/foo.tif` writes to
  `OUTPUT_PATH/Oxygen/600V_O/foo_*`).

In [ ]:
INPUT_PATH  = r"../../data/images"    # TIFF image or folder of TIFFs
OUTPUT_PATH = r"../../outputs/annotations"    # folder where masks, annotations, and manifest are saved

print(f"Input path : {INPUT_PATH or '(not set)'}")
print(f"Output path: {OUTPUT_PATH or '(not set)'}")

## Multi-class Annotator Code

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Callable

import napari
import numpy as np
import tifffile
from skimage.draw import polygon as draw_polygon
from qtpy.QtCore import Qt
from qtpy.QtWidgets import (
    QButtonGroup,
    QGroupBox,
    QHBoxLayout,
    QLabel,
    QPushButton,
    QRadioButton,
    QVBoxLayout,
    QWidget,
)

CLASSES = {
    1: {"name": "nucleus", "color": "cyan"},
    2: {"name": "full_cell", "color": "green"},
    3: {"name": "lysosome", "color": "red"},
}

_COLOR_HEX = {
    "cyan": "#00FFFF",
    "green": "#00FF00",
    "red": "#FF4444",
}

_TIFF_EXTS = {".tif", ".tiff"}


def _layer_to_annotations(layer, class_id: int) -> list[dict]:
    annotations = []
    for shape_data in layer.data:
        coords = np.array(shape_data)
        if len(coords) < 3:
            continue
        y_coords = coords[:, 0]
        x_coords = coords[:, 1]
        seg: list[float] = []
        for y, x in zip(y_coords, x_coords):
            seg.extend([float(x), float(y)])
        x_min, x_max = float(x_coords.min()), float(x_coords.max())
        y_min, y_max = float(y_coords.min()), float(y_coords.max())
        annotations.append({
            "segmentation": [seg],
            "bbox": [x_min, y_min, x_max - x_min, y_max - y_min],
            "area": float((x_max - x_min) * (y_max - y_min)),
            "category_id": class_id,
            "category_name": CLASSES[class_id]["name"],
        })
    return annotations


def resolve_out_dir(
    image_path: Path,
    output_path: Path,
    input_root: Path | None,
) -> Path:
    """Per-image output directory, mirroring image's subfolder under input_root.

    If ``input_root`` is a directory and ``image_path`` lives under it,
    the result is ``output_path / image_path.parent.relative_to(input_root)``.
    Otherwise it is ``output_path`` itself.  Created if missing.
    """
    rel = Path()
    if input_root is not None and input_root.is_dir():
        try:
            rel = image_path.parent.resolve().relative_to(input_root.resolve())
        except ValueError:
            rel = Path()
    out_dir = output_path / rel
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir


def save_annotations_json(
    class_layers: dict, image_name: str, out_dir: Path,
) -> Path:
    """Write the COCO-style polygon JSON for one image into ``out_dir``."""
    all_anns = []
    for cid, layer in class_layers.items():
        all_anns.extend(_layer_to_annotations(layer, cid))
    out = {
        "image": image_name,
        "annotations": all_anns,
        "categories": [
            {"id": cid, "name": ci["name"]} for cid, ci in CLASSES.items()
        ],
    }
    stem = Path(image_name).stem
    json_path = out_dir / f"{stem}_annotations.json"
    json_path.write_text(json.dumps(out, indent=2), encoding="utf-8")
    print(f"Saved {len(all_anns)} annotations -> {json_path}")
    return json_path


def _rasterize_class_layer(
    layer, shape: tuple[int, int]
) -> tuple[np.ndarray, int]:
    """Rasterize all polygons in a class layer into a single binary mask.

    Returns (mask, n_instances) where mask is a uint8 array of `shape`
    with 1 inside any polygon and 0 elsewhere.  Polygons with fewer
    than 3 vertices are skipped.
    """
    mask = np.zeros(shape, dtype=np.uint8)
    n = 0
    for shape_data in layer.data:
        coords = np.asarray(shape_data, dtype=float)
        if coords.ndim != 2 or coords.shape[0] < 3:
            continue
        ys = coords[:, 0]
        xs = coords[:, 1]
        rr, cc = draw_polygon(ys, xs, shape=shape)
        if rr.size == 0:
            continue
        mask[rr, cc] = 1
        n += 1
    return mask, n


def save_class_masks(
    class_layers: dict,
    image_path: Path,
    image_shape: tuple[int, int],
    out_dir: Path,
) -> Path:
    """Rasterize polygon annotations into per-class binary TIF masks.

    Writes into ``out_dir``::

        <out_dir>/<stem>_<class>.tif        (uint8, 0 / 255)
        <out_dir>/<stem>_manifest.json

    Only classes with at least one drawn polygon (and a non-empty
    rasterized mask) get a TIF.  ``cytosol`` is derived as
    ``full_cell AND NOT nucleus`` and only written when both
    ``full_cell`` and ``nucleus`` produced non-empty masks.

    The manifest always reports ``n_instances``, ``n_pixels``, and
    ``charging_pct`` (0.0 for manual annotation) for all four classes,
    plus ``has_charging: false``.
    """
    stem = image_path.stem

    masks: dict[str, np.ndarray] = {}
    counts: dict[str, int] = {}
    for cid, cinfo in CLASSES.items():
        name = cinfo["name"]
        mask, n = _rasterize_class_layer(class_layers[cid], image_shape)
        masks[name] = mask
        counts[name] = n

    nuc = masks.get("nucleus")
    fc = masks.get("full_cell")
    if (
        nuc is not None and fc is not None
        and nuc.any() and fc.any()
    ):
        cyto = (fc.astype(bool) & ~nuc.astype(bool)).astype(np.uint8)
    else:
        cyto = np.zeros(image_shape, dtype=np.uint8)
    masks["cytosol"] = cyto
    counts["cytosol"] = 1 if cyto.any() else 0

    manifest: dict = {}
    written: list[str] = []
    for name in ("nucleus", "full_cell", "lysosome", "cytosol"):
        mask = masks[name]
        n_inst = counts[name]
        n_px = int(mask.sum())
        manifest[name] = {
            "n_instances": n_inst,
            "n_pixels": n_px,
            "charging_pct": 0.0,
        }
        if n_inst > 0 and n_px > 0:
            tif_path = out_dir / f"{stem}_{name}.tif"
            tifffile.imwrite(str(tif_path), (mask * 255).astype(np.uint8))
            written.append(name)
    manifest["has_charging"] = False

    manifest_path = out_dir / f"{stem}_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

    summary = ", ".join(written) if written else "(none)"
    print(f"Saved masks [{summary}] -> {out_dir}")
    return out_dir


def save_image_outputs(
    class_layers: dict,
    image_path: Path,
    image_shape: tuple[int, int],
    output_path: Path,
    input_root: Path | None,
) -> Path:
    """Write JSON + per-class TIFs + manifest for one image into the
    appropriate per-image folder under ``output_path``.
    """
    out_dir = resolve_out_dir(image_path, output_path, input_root)
    save_annotations_json(class_layers, image_path.name, out_dir)
    save_class_masks(class_layers, image_path, image_shape, out_dir)
    return out_dir


def load_annotations(json_path: Path) -> list[dict]:
    data = json.loads(json_path.read_text(encoding="utf-8"))
    return data.get("annotations", [])


def _load_into_layers(annotations: list[dict], class_layers: dict):
    per_class: dict[int, list[np.ndarray]] = {cid: [] for cid in CLASSES}
    for ann in annotations:
        cid = ann.get("category_id", 0)
        if cid not in CLASSES:
            continue
        seg = ann.get("segmentation", [[]])
        flat = seg[0] if seg else []
        if len(flat) < 6:
            continue
        pts = []
        for j in range(0, len(flat), 2):
            x, y = flat[j], flat[j + 1]
            pts.append([y, x])
        per_class[cid].append(np.array(pts))

    for cid, layer in class_layers.items():
        polys = per_class.get(cid, [])
        if polys:
            layer.data = polys
            layer.shape_type = ["polygon"] * len(polys)
        else:
            layer.data = []


def _clear_all_layers(class_layers: dict):
    for layer in class_layers.values():
        layer.data = []


def _build_widget(
    viewer,
    class_layers,
    nav_state,
    output_path: Path,
    input_root: Path | None,
    get_image_shape: Callable[[], tuple[int, int]],
):
    widget = QWidget()
    layout = QVBoxLayout()
    layout.setSpacing(8)
    widget.setLayout(layout)

    image_list = nav_state["image_list"]
    total_images = len(image_list)

    if total_images > 1:
        nav_box = QGroupBox("Navigation")
        nav_vlayout = QVBoxLayout()
        nav_vlayout.setSpacing(4)
        nav_box.setLayout(nav_vlayout)

        nav_label = QLabel("")
        nav_label.setAlignment(Qt.AlignCenter)
        nav_label.setStyleSheet("font-size: 13px; font-weight: bold; padding: 4px;")

        def _update_nav_label():
            idx = nav_state["index"]
            cur = image_list[idx]
            cur_dir = resolve_out_dir(cur, output_path, input_root)
            ann_path = cur_dir / f"{cur.stem}_annotations.json"
            marker = "  [annotated]" if ann_path.exists() else ""
            nav_label.setText(
                f"<b>{cur.stem}</b>{marker}<br>Image {idx + 1} of {total_images}"
            )

        _update_nav_label()
        nav_vlayout.addWidget(nav_label)

        btn_row = QHBoxLayout()
        prev_btn = QPushButton("Prev")
        next_btn = QPushButton("Next")
        for b in (prev_btn, next_btn):
            b.setStyleSheet(
                "QPushButton { font-size: 13px; padding: 8px; "
                "background-color: #555; color: white; border-radius: 4px; }"
                "QPushButton:hover { background-color: #777; }"
                "QPushButton:disabled { background-color: #333; color: #666; }"
            )

        def _update_nav_buttons():
            prev_btn.setEnabled(nav_state["index"] > 0)
            next_btn.setEnabled(nav_state["index"] < total_images - 1)

        _update_nav_buttons()

        def _on_prev():
            if nav_state["index"] > 0:
                nav_state["load_image"](nav_state["index"] - 1)
                _update_nav_label()
                _update_nav_buttons()
                _refresh_counts()

        def _on_next():
            if nav_state["index"] < total_images - 1:
                nav_state["load_image"](nav_state["index"] + 1)
                _update_nav_label()
                _update_nav_buttons()
                _refresh_counts()

        prev_btn.clicked.connect(_on_prev)
        next_btn.clicked.connect(_on_next)
        btn_row.addWidget(prev_btn)
        btn_row.addWidget(next_btn)
        nav_vlayout.addLayout(btn_row)
        layout.addWidget(nav_box)
    else:
        title = QLabel(f"<b>{image_list[0].name}</b>")
        title.setAlignment(Qt.AlignCenter)
        title.setStyleSheet("font-size: 13px; padding: 4px;")
        layout.addWidget(title)
        _update_nav_label = lambda: None

    class_box = QGroupBox("Active class  (selects layer)")
    class_layout = QVBoxLayout()
    class_layout.setSpacing(4)
    class_box.setLayout(class_layout)
    btn_group = QButtonGroup(widget)

    radio_buttons: dict[int, QRadioButton] = {}
    for cid, cinfo in CLASSES.items():
        hex_col = _COLOR_HEX.get(cinfo["color"], "#FFF")
        rb = QRadioButton(f"  {cid}. {cinfo['name']}")
        rb.setStyleSheet(
            f"QRadioButton {{ font-size: 14px; font-weight: bold; "
            f"color: {hex_col}; padding: 6px; }}"
            f"QRadioButton::indicator {{ width: 18px; height: 18px; }}"
        )
        btn_group.addButton(rb, cid)
        class_layout.addWidget(rb)
        radio_buttons[cid] = rb

    radio_buttons[1].setChecked(True)

    def _on_class_changed(btn_id: int):
        for cid, layer in class_layers.items():
            if cid == btn_id:
                viewer.layers.selection.active = layer
                layer.mode = "add_polygon"
            else:
                layer.mode = "pan_zoom"
        viewer.status = f"Active: {btn_id} = {CLASSES[btn_id]['name']}"
        _refresh_counts()

    btn_group.idClicked.connect(_on_class_changed)
    layout.addWidget(class_box)

    counts_label = QLabel("")
    counts_label.setStyleSheet("font-size: 12px; padding: 4px;")
    counts_label.setWordWrap(True)
    layout.addWidget(counts_label)

    def _refresh_counts():
        lines = []
        total = 0
        for cid, cinfo in CLASSES.items():
            n = len(class_layers[cid].data)
            total += n
            hex_col = _COLOR_HEX.get(cinfo["color"], "#FFF")
            lines.append(
                f"<span style='color:{hex_col}'>{cinfo['name']}: <b>{n}</b></span>"
            )
        counts_label.setText(
            f"Annotations ({total} total):<br>" + " &nbsp;|&nbsp; ".join(lines)
        )

    _refresh_counts()

    act_box = QGroupBox("Actions")
    act_layout = QVBoxLayout()
    act_layout.setSpacing(4)
    act_box.setLayout(act_layout)

    status_label = QLabel("")
    status_label.setStyleSheet("font-size: 11px; color: #aaa; padding: 2px;")

    save_btn = QPushButton("Save annotations")
    save_btn.setStyleSheet(
        "QPushButton { font-size: 13px; font-weight: bold; padding: 8px; "
        "background-color: #2d7d46; color: white; border-radius: 4px; }"
        "QPushButton:hover { background-color: #3a9d5a; }"
    )

    def _on_save():
        cur = image_list[nav_state["index"]]
        try:
            out_dir = save_image_outputs(
                class_layers, cur, get_image_shape(), output_path, input_root,
            )
            status_label.setText(f"Saved -> {out_dir}")
        except Exception as exc:
            status_label.setText(f"Save FAILED: {exc}")
        _refresh_counts()
        _update_nav_label()

    save_btn.clicked.connect(_on_save)
    act_layout.addWidget(save_btn)

    load_btn = QPushButton("Load annotations")
    load_btn.setStyleSheet(
        "QPushButton { font-size: 13px; padding: 8px; "
        "background-color: #2d5a7d; color: white; border-radius: 4px; }"
        "QPushButton:hover { background-color: #3a7a9d; }"
    )

    def _on_load():
        cur = image_list[nav_state["index"]]
        cur_dir = resolve_out_dir(cur, output_path, input_root)
        jp = cur_dir / f"{cur.stem}_annotations.json"
        if jp.exists():
            anns = load_annotations(jp)
            _load_into_layers(anns, class_layers)
            status_label.setText(f"Loaded {len(anns)} annotations")
        else:
            status_label.setText("No saved annotations found")
        _refresh_counts()

    load_btn.clicked.connect(_on_load)
    act_layout.addWidget(load_btn)

    del_btn = QPushButton("Delete selected polygon")
    del_btn.setStyleSheet(
        "QPushButton { font-size: 13px; padding: 8px; "
        "background-color: #7d2d2d; color: white; border-radius: 4px; }"
        "QPushButton:hover { background-color: #9d3a3a; }"
    )

    def _on_delete():
        active = viewer.layers.selection.active
        if active is None or active.name == "image":
            status_label.setText("Select a polygon on a class layer first")
            return
        sel = list(active.selected_data)
        if not sel:
            status_label.setText("No polygon selected")
            return
        data_list = list(active.data)
        for idx in sorted(sel, reverse=True):
            if idx < len(data_list):
                data_list.pop(idx)
        active.data = data_list if data_list else []
        active.selected_data = set()
        status_label.setText(f"Deleted {len(sel)} polygon(s)")
        _refresh_counts()

    del_btn.clicked.connect(_on_delete)
    act_layout.addWidget(del_btn)
    act_layout.addWidget(status_label)
    layout.addWidget(act_box)

    help_label = QLabel(
        "<b>How to annotate:</b><br>"
        "1. Select a class above (switches the active layer)<br>"
        "2. Click vertices around the region boundary<br>"
        "3. Double-click to close the polygon<br>"
        "4. Switch class and repeat<br>"
        "5. Click <b>Save</b> when done<br>"
        "6. Click <b>Next</b> to go to the next image"
    )
    help_label.setWordWrap(True)
    help_label.setStyleSheet("font-size: 11px; color: #ccc; padding: 6px;")
    layout.addWidget(help_label)

    layout.addStretch()

    for layer in class_layers.values():
        layer.events.data.connect(lambda e: _refresh_counts())

    return widget


print('Multi-class annotator ready.')

## Launch Viewer

In [ ]:
if not INPUT_PATH:
    INPUT_PATH = input("Enter path to a TIFF image or folder: ")
if not OUTPUT_PATH:
    raise ValueError("OUTPUT_PATH must be set")

input_path = Path(INPUT_PATH)
output_path = Path(OUTPUT_PATH)
output_path.mkdir(parents=True, exist_ok=True)

input_root: Path | None = input_path if input_path.is_dir() else None

if input_path.is_dir():
    image_list = sorted(
        [f for f in input_path.rglob("*") if f.suffix.lower() in _TIFF_EXTS],
        key=lambda p: (str(p.parent), p.name),
    )
    if not image_list:
        raise FileNotFoundError(f"No TIFF files under {input_path}")
else:
    image_list = [input_path]

start_idx = 0
if not input_path.is_dir():
    for i, p in enumerate(image_list):
        if p == input_path:
            start_idx = i
            break

print(f"Input  : {input_path}  ({len(image_list)} image(s))")
print(f"Output : {output_path}")

from napari.layers import Image as _ImageLayer

viewer = napari.Viewer(title=f"Annotate -- {image_list[start_idx].name}")

img_state: dict = {"layer": None}


def _open_image_via_napari(path: Path) -> _ImageLayer:
    """Open `path` through napari's plugin reader chain (same as drag-and-drop).

    Removes any existing Image layers, then routes the read through
    ``viewer.open`` so contrast limits, dtype handling, RGB/multi-channel
    detection, etc. match the drag-and-drop behavior exactly.  The new
    image layer is moved to the bottom so the Shapes layers stay on top.
    """
    for lyr in list(viewer.layers):
        if isinstance(lyr, _ImageLayer):
            viewer.layers.remove(lyr)

    before = {id(L) for L in viewer.layers}
    result = viewer.open(str(path))
    added: list = []
    if isinstance(result, list):
        added = [L for L in result if isinstance(L, _ImageLayer)]
    if not added:
        added = [
            L for L in viewer.layers
            if id(L) not in before and isinstance(L, _ImageLayer)
        ]
    if not added:
        raise RuntimeError(f"napari did not add an Image layer for {path}")

    new_img = added[0]
    idx = viewer.layers.index(new_img)
    if idx != 0:
        viewer.layers.move(idx, 0)
    return new_img


img_state["layer"] = _open_image_via_napari(image_list[start_idx])

class_layers = {}
for cid, cinfo in CLASSES.items():
    layer = viewer.add_shapes(
        name=f"{cid}_{cinfo['name']}",
        shape_type="polygon",
        edge_color=cinfo["color"],
        face_color="transparent",
        edge_width=2,
    )
    class_layers[cid] = layer

first_cid = list(CLASSES.keys())[0]
viewer.layers.selection.active = class_layers[first_cid]
class_layers[first_cid].mode = "add_polygon"

nav_state = {"image_list": image_list, "index": start_idx, "load_image": None}

def _get_image_shape() -> tuple[int, int]:
    layer = img_state["layer"]
    arr = layer.data
    if getattr(layer, "rgb", False) and arr.ndim >= 3:
        return tuple(arr.shape[:2])
    return tuple(arr.shape[-2:])


def _load_image(new_idx):
    cur = image_list[nav_state["index"]]
    has_data = any(len(l.data) > 0 for l in class_layers.values())
    if has_data:
        try:
            save_image_outputs(
                class_layers, cur, _get_image_shape(), output_path, input_root,
            )
        except Exception as exc:
            print(f"Save FAILED for {cur.name}: {exc}")
    nav_state["index"] = new_idx
    new_path = image_list[new_idx]
    img_state["layer"] = _open_image_via_napari(new_path)
    _clear_all_layers(class_layers)
    new_dir = resolve_out_dir(new_path, output_path, input_root)
    ann_file = new_dir / f"{new_path.stem}_annotations.json"
    if ann_file.exists():
        anns = load_annotations(ann_file)
        _load_into_layers(anns, class_layers)
        print(f"Loaded annotations for {new_path.name}")
    viewer.title = f"Annotate -- {new_path.name}"
    viewer.reset_view()

nav_state["load_image"] = _load_image

ctrl = _build_widget(
    viewer, class_layers, nav_state,
    output_path=output_path,
    input_root=input_root,
    get_image_shape=_get_image_shape,
)
viewer.window.add_dock_widget(ctrl, name="Annotation Controls", area="right")

start_dir = resolve_out_dir(image_list[start_idx], output_path, input_root)
ann_file = start_dir / f"{image_list[start_idx].stem}_annotations.json"
if ann_file.exists():
    anns = load_annotations(ann_file)
    _load_into_layers(anns, class_layers)
    print(f"Auto-loaded annotations for {image_list[start_idx].name}")

print("Napari viewer opened with Multi-class Annotator widget.")